# 11 - Adversarial Robustness (Objective 4)

The novel contribution. Two attacks, run against the same models on the same frozen test set.

**`romanisation_attack`** rewrites tokens into plausible romanisation variants using the rules
in `variants.py`. Every output is a spelling a Hindi speaker could produce naturally and read
without difficulty.

**`character_attack`** is the control, in the spirit of AdvCodeMix: random character swaps,
substitutions, deletions and repetitions. Not linguistically motivated.

**Why both.** If the two attacks degrade a model equally at the same realised edit rate, the
linguistic rules add nothing over generic noise. If the romanisation attack degrades it more,
the structure matters - and that is the claim the dissertation makes.

**Setup before running:** copy `attack.py` into `notebooks/hinglish_hate/`, then add to
`__init__.py`:

```python
from .attack import romanisation_attack, character_attack, attack_corpus, attack_report
__all__ += ["romanisation_attack", "character_attack", "attack_corpus", "attack_report"]
```

### 1. Setup

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('local / already mounted:', e)

import sys; sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score

from hinglish_hate import load_lexicon, filter_romanised, load_hasoc2022_threads
from hinglish_hate.baseline import build_full_pipeline
from hinglish_hate.attack import (romanisation_attack, character_attack,
                                  attack_corpus, attack_report)

DATA = Path('/content/drive/MyDrive/dissertation/data')
W    = Path('/content/drive/MyDrive/dissertation/writing'); W.mkdir(parents=True, exist_ok=True)
FIG  = W/'figures'; FIG.mkdir(exist_ok=True)

train_df = pd.read_parquet(DATA/'splits'/'bohra_train.parquet')
test_df  = pd.read_parquet(DATA/'splits'/'bohra_test.parquet')
print(f'train {len(train_df)} | test {len(test_df)}')

### 2. See what the attacks actually do

Always eyeball the perturbations before trusting a number. If the romanisation output is not
readable to you as a Hindi speaker, the rules are too aggressive and the attack is not a
realistic evasion - it is just noise with extra steps.

**This is a judgement only you can make**, and it is worth a paragraph in the write-up.

In [ ]:
samples = test_df['text'].head(6).tolist()
for s in samples:
    print('ORIGINAL   :', s[:100])
    print('romanisation:', romanisation_attack(s, rate=0.5, seed=42)[:100])
    print('character   :', character_attack(s, rate=0.5, seed=42)[:100])
    print()

### 3. Fit the baseline on clean training data

The threat model matters and should be stated explicitly: the attacker perturbs the **input at
test time only**. The model is trained on clean text and never sees perturbed examples. That
is the realistic case - a deployed moderation classifier does not get retrained the moment
someone invents a new spelling.

In [ ]:
terms = load_lexicon(DATA/'hinglish_slur_lexicon.csv')
pipe  = build_full_pipeline(terms)
pipe.fit(train_df['text'], train_df['label'])

clean_pred  = pipe.predict(test_df['text'])
clean_macro = f1_score(test_df['label'], clean_pred, average='macro')
clean_hate  = f1_score(test_df['label'], clean_pred, pos_label=1, zero_division=0)
print(f'clean: macro-F1 {clean_macro:.3f} | hate-F1 {clean_hate:.3f}')

### 4. Attack across a range of edit rates

A single rate gives one number. A curve shows how quickly the model degrades, which is the
more informative result and makes a better figure.

In [ ]:
rows = []
for rate in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    for name, fn in [('romanisation', romanisation_attack), ('character', character_attack)]:
        if rate == 0.0:
            rows.append({'attack':name,'requested_rate':0.0,'realised_rate':0.0,
                         'macro_f1':clean_macro,'hate_f1':clean_hate})
            continue
        adv  = attack_corpus(test_df, fn, rate=rate, seed=42)
        pred = pipe.predict(adv['text'])
        rows.append({'attack':name, 'requested_rate':rate,
                     'realised_rate':round(adv['token_change_rate'].mean(),3),
                     'macro_f1':f1_score(adv['label'], pred, average='macro'),
                     'hate_f1':f1_score(adv['label'], pred, pos_label=1, zero_division=0)})
        print(f"  {name:12s} rate {rate:.2f} (realised {rows[-1]['realised_rate']:.3f}) "
              f"-> macro-F1 {rows[-1]['macro_f1']:.3f}")

curve = pd.DataFrame(rows)
curve.to_csv(W/'attack_curve_baseline.csv', index=False)
display(curve.round(3))

### 5. The degradation curve

In [ ]:
fig, ax = plt.subplots(figsize=(7.5,4.5))
for name, marker in [('romanisation','o'), ('character','s')]:
    sub = curve[curve['attack']==name].sort_values('requested_rate')
    ax.plot(sub['requested_rate'], sub['macro_f1'], marker=marker, lw=2, label=name)
ax.axhline(clean_macro, color='grey', ls='--', lw=1.2, label='clean')
ax.set_xlabel('requested token perturbation rate'); ax.set_ylabel('macro-F1')
ax.set_ylim(0, 0.75); ax.grid(alpha=.3); ax.legend()
ax.set_title('Statistical baseline under adversarial perturbation\n(trained on clean text, attacked at test time)')
plt.tight_layout(); plt.savefig(FIG/'fig3_attack_curve.png', dpi=150, bbox_inches='tight'); plt.show()

r50 = curve[(curve['attack']=='romanisation') & (curve['requested_rate']==0.5)].iloc[0]
c50 = curve[(curve['attack']=='character')    & (curve['requested_rate']==0.5)].iloc[0]
print(attack_report(clean_macro, r50['macro_f1'], 'romanisation @ 0.5'))
print(attack_report(clean_macro, c50['macro_f1'], 'character @ 0.5'))
print('\nCompare at *realised* rates, not requested - the two attacks do not edit equally.')

### 6. Next

- Run the same attack against XLM-R, MuRIL and IndicBERT. The prediction worth testing: the
  character n-gram baseline should be *more* robust than the transformers, because sub-word
  shape survives a spelling change better than a fixed learned vocabulary does.
- **Attack the cross-dataset setting too.** Objectives 3 and 4 compound: a model that already
  fails to transfer, then gets attacked, is the realistic worst case.
- Adversarial fine-tuning as a defence: train on clean plus perturbed data, re-measure.
- Report every attack result in `all_results.csv`, same as the model runs.